# Qwen-Image-2.1 Turbo — نسخهٔ بدون سانسور روی کولب

**قبل از Run all:** `Runtime → Change runtime type → T4 GPU`.

تولید و ادیت در یک مدل ۷B. انکودر متن «هریتیک» (abliterated) است، پس پرامپت را رد نمی‌کند.

| قطعه | فایل | حجم |
|---|---|---:|
| DiT (turbo، ۸ استپ) | `qwen-image-2.1-turbo-Q4_K_M.gguf` (abenzerps) | 4.05 GB |
| Text encoder هرتیک | `qwen3vl_8b_heretic-Q4_K_M.gguf` (pottokao) | 5.03 GB |
| Vision tower انکودر | `mmproj-qwen3vl_8b_heretic-f16.gguf` | 1.16 GB |
| VAE | `qwen_image_2.1_vae_bf16.safetensors` | 0.68 GB |
| LoRA محتوایی (اختیاری) | `NSFW Qwen Lora.safetensors` (f23gg) | 0.16 GB |

جمع ≈ ۱۱ گیگ → روی T4 ۱۶ گیگی جا می‌شود.

> انکودر هریتیک **باید** همراهش فایل `mmproj` باشد؛ نام هر دو را عوض نکن (پچ، mmproj را از روی نام پیدا می‌کند).
> روی A100/L4 همین نوتبوک را با `Q8_0` برای DiT و انکودر اجرا کن (متغیر `QUANT` در سل ۳).

In [ ]:
# ۱) بررسی GPU و دیسک
import shutil, subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip() or "NO GPU -> Runtime > Change runtime type > T4 GPU")
import torch; print("torch", torch.__version__, "cuda:", torch.cuda.is_available())
t, u, f = shutil.disk_usage("/content"); print(f"disk free: {f/1e9:.1f} GB")

In [ ]:
%%bash
set -e
cd /content
[ -d ComfyUI ] || git clone -q --depth 1 https://github.com/comfyanonymous/ComfyUI.git
cd ComfyUI
pip install -q -r requirements.txt
cd custom_nodes
[ -d ComfyUI-GGUF ]            || git clone -q --depth 1 https://github.com/city96/ComfyUI-GGUF
[ -d ComfyUI-GGUF-Qwen3VL-TE ] || git clone -q --depth 1 https://github.com/pottokao-dotcom/ComfyUI-GGUF-Qwen3VL-TE
pip install -q -r ComfyUI-GGUF/requirements.txt
pip install -q hf_transfer
echo "--- custom_nodes ---"; ls

In [ ]:
# ۳) دانلود مدل‌ها از HF (no token needed - همه public)
import os, sys
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
from huggingface_hub import hf_hub_download

QUANT = "Q4_K_M"          # T4 -> Q4_K_M | A100/L4 -> Q8_0
M = "/content/ComfyUI/models"
TE_Q = "Q8_0" if QUANT == "Q8_0" else QUANT

jobs = [
    ("abenzerps/Qwen-Image-2.1-Turbo-Quantized", f"qwen-image-2.1-turbo-{QUANT}.gguf", f"{M}/unet"),
    ("pottokao/Qwen-Image-2.1-Text-Encoder-Heretic-GGUF", f"qwen3vl_8b_heretic-{TE_Q}.gguf", f"{M}/text_encoders"),
    ("pottokao/Qwen-Image-2.1-Text-Encoder-Heretic-GGUF", "mmproj-qwen3vl_8b_heretic-f16.gguf", f"{M}/text_encoders"),
    ("abenzerps/Qwen-Image-2.1-Turbo-Quantized", "vae/qwen_image_2.1_vae_bf16.safetensors", f"{M}/vae"),
    ("f23gg/NSFW-LORA-Qwen-Image-2.1", "NSFW Qwen Lora.safetensors", f"{M}/loras"),
]
for repo, fn, dest in jobs:
    os.makedirs(dest, exist_ok=True)
    p = hf_hub_download(repo_id=repo, filename=fn, local_dir=dest)
    print(f"{os.path.getsize(p)/1e9:6.2f} GB  {p}", flush=True)
print("done")

In [ ]:
# ۴) اجرای ComfyUI + لینک عمومی
import subprocess, sys, threading, time
proc = subprocess.Popen([sys.executable, "main.py", "--share", "--port", "8188"],
                        cwd="/content/ComfyUI", stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
def tail():
    for line in proc.stdout:
        sys.stdout.write(line); sys.stdout.flush()
threading.Thread(target=tail, daemon=True).start()
time.sleep(75)   # تا لینک gradio.live بالا بیاید
print("\n>>> آدرس GUI چند خط بالاتر، بعد از «To see the GUI go to» است.")

## تنظیمات سَمپلر (در ComfyUI)

**گراف:** `Unet Loader (GGUF)` → `qwen-image-2.1-turbo-Q4_K_M.gguf` · `CLIPLoader (GGUF)` با type = `qwen_image` → `qwen3vl_8b_heretic-Q4_K_M.gguf` · `VAELoader` → vae.
(یا از `Workflow → Browse Templates → Qwen-Image` تمپلیت را بگیر و فقط نود لودر DiT را با نسخهٔ GGUF عوض کن.)

- **۸ استپ**، `cfg 1.0`، sampler `euler`، scheduler `simple`، رزولوشن 1024×1024 (تا 1328). بیشتر از ۸ استپ لازم نیست — مدل turbo است.
- لورای محتوایی: `LoraLoaderModelOnly` با `NSFW Qwen Lora.safetensors` روی **0.8**.
- **متن داخل تصویر** (تابلو/برچسب): دو نود `KSamplerAdvanced` زنجیره‌ای — اولی `cfg 1.0`، `start 0 → end 15`، `return_with_leftover_noise=enable`؛ دومی `add_noise=disable`، `cfg 3.0`، `start 15 → end 25`. یک seed، یک پاس.
- **OOM روی T4؟** سل ۴ را با `--lowvram` اجرا کن (کندتر، ولی ۱۰۰٪ اجرا می‌شود).
- ادیت دستوری (بدون ماسک) همان مدل است: تمپلیت Qwen-Image Edit + لودر GGUF؛ ۱ تا ۳ تصویر مرجع.

**نکته:** ادیت این مدل ماسک ندارد — کاملاً دستوری است. برای اصلاح نقطه‌ای دقیق مناسب نیست.